# Prompt 20: DataFrame Partitioning — Repartition and Coalesce
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (30%)

---

## What Is a Partition?

A Spark DataFrame is split into **partitions** — chunks of data that can be processed in parallel across executor cores. Each task processes one partition at a time.

```
DataFrame
├── Partition 0  → Task 0 on Executor A
├── Partition 1  → Task 1 on Executor B
├── Partition 2  → Task 2 on Executor A
└── Partition 3  → Task 3 on Executor B
```

Default partition count after reading is controlled by:
- `spark.sql.shuffle.partitions` (default **200**) — used after shuffles
- `spark.default.parallelism` — used for RDD operations
- Number of input file splits for reads

---

## The Three Partitioning Operations

| Operation | Shuffle? | Effect on count | Use case |
|-----------|----------|-----------------|----------|
| `repartition(n)` | **Yes** — full shuffle | Can increase **or** decrease | Increase parallelism; balance partitions |
| `repartition(n, col)` | **Yes** — full shuffle | Can increase **or** decrease | Co-locate rows by column value (e.g., before a join) |
| `coalesce(n)` | **No** — merge local partitions | Only **decrease** | Reduce partition count cheaply; produce fewer output files |

**Critical exam rule:** `coalesce` can ONLY decrease partition count. To increase, use `repartition`.

---

## repartition vs coalesce — Shuffle Trade-off

```
repartition(3):
  P0 ──┐
  P1 ──┼──► SHUFFLE ──► P0' (balanced)
  P2 ──┤              ► P1' (balanced)
  P3 ──┘              ► P2' (balanced)

coalesce(2):           (no shuffle — merges on same executor when possible)
  P0 ─┐
  P1 ─┴──► P0' (P0 + P1 merged)
  P2 ─┐
  P3 ─┴──► P1' (P2 + P3 merged)
```

Because `coalesce` avoids the shuffle it is much cheaper than `repartition` when reducing partition count. However, the merged partitions may be **unequally sized** (skewed), since rows are merged as-is without redistribution.

---

## write.partitionBy vs repartition in Memory

| | `write.partitionBy('col')` | `repartition(n, 'col')` |
|--|---------------------------|-------------------------|
| Where it applies | **Storage** — directory hierarchy on disk | **Memory** — in-memory data distribution |
| Output directories | `/year=2024/month=1/` etc. | n files, not by column value |
| Partition pruning | **Yes** — avoids reading irrelevant dirs | No |
| Shuffle | Implicit on write | Yes — full shuffle |
| Common use | Read-optimisation for downstream queries | Pre-shuffle for join/agg performance |

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, spark_partition_id

spark = SparkSession.builder \
    .appName('RepartitionCoalesce') \
    .master('local[4]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Helper — show how many rows are in each partition
def show_partition_sizes(df, label):
    sizes = df.withColumn('pid', spark_partition_id()) \
              .groupBy('pid').count() \
              .orderBy('pid')
    print(f'--- {label} ---')
    print(f'Num partitions: {df.rdd.getNumPartitions()}')
    sizes.show(truncate=False)

# Sample DataFrame — 100 rows, 4 categories
data = [(i, ['Electronics','Clothing','Books','Food'][i % 4], float(i * 10))
        for i in range(1, 101)]
df = spark.createDataFrame(data, ['id', 'category', 'amount'])

print(f'Default partition count: {df.rdd.getNumPartitions()}')
show_partition_sizes(df, 'Original')

In [ ]:
# Cell 2: repartition(n) — full shuffle, balanced output

print('=== repartition(n) — increase partitions ===')
df_more = df.repartition(10)
show_partition_sizes(df_more, 'repartition(10) — increased')

print('=== repartition(n) — decrease partitions ===')
df_less = df.repartition(2)
show_partition_sizes(df_less, 'repartition(2) — decreased')

print('=== repartition(1) — all rows in one partition ===')
df_one = df.repartition(1)
print(f'Partitions after repartition(1): {df_one.rdd.getNumPartitions()}')  # 1

print('=== repartition(n, col) — co-locate rows by column value ===')
# All rows with the same category go to the same partition
df_by_cat = df.repartition(4, 'category')
show_partition_sizes(df_by_cat, 'repartition(4, category)')

# Verify each partition contains only one category
print('Categories per partition after repartition(4, category):')
df_by_cat.withColumn('pid', spark_partition_id()) \
         .groupBy('pid', 'category').count() \
         .orderBy('pid', 'category') \
         .show()

print('=== repartition with multiple columns ===')
df_multi = df.repartition(8, 'category', 'id')
print(f'Partitions: {df_multi.rdd.getNumPartitions()}')
show_partition_sizes(df_multi, 'repartition(8, category, id)')

In [ ]:
# Cell 3: coalesce(n) — no shuffle, merge local partitions

print(f'Starting partition count: {df.rdd.getNumPartitions()}')

print('=== coalesce(2) — reduce without shuffle ===')
df_coal2 = df.coalesce(2)
show_partition_sizes(df_coal2, 'coalesce(2)')
# Note: partitions may be uneven — no redistribution of rows

print('=== coalesce(1) — single partition (single output file on write) ===')
df_coal1 = df.coalesce(1)
print(f'Partitions after coalesce(1): {df_coal1.rdd.getNumPartitions()}')  # 1

print('=== IMPORTANT: coalesce cannot increase partition count ===')
# Requesting MORE partitions than currently exist does nothing (no error)
current = df.rdd.getNumPartitions()
target  = current + 100
df_attempt = df.coalesce(target)
print(f'Current: {current}, requested: {target}, got: {df_attempt.rdd.getNumPartitions()}')
# Result: same as current — coalesce silently ignores the increase request

print('=== To increase, you MUST use repartition ===')
df_increased = df.repartition(target)
print(f'repartition({target}) result: {df_increased.rdd.getNumPartitions()}')

print('=== repartition vs coalesce — balance comparison ===')
# repartition produces equally-sized partitions (hash-based shuffle)
df_repart3 = df.repartition(3)
show_partition_sizes(df_repart3, 'repartition(3) — balanced')

# coalesce may produce skewed partitions (merges as-is)
df_coal3 = df.coalesce(3)
show_partition_sizes(df_coal3, 'coalesce(3) — may be skewed')

In [ ]:
# Cell 4: write.partitionBy vs repartition — output file count

import tempfile, os
BASE = tempfile.mkdtemp().replace('\\', '/')

print('=== Default write — one file per in-memory partition ===')
default_path = f'{BASE}/default'
df.write.mode('overwrite').parquet(default_path)
n_files = len([f for f in os.listdir(default_path) if f.endswith('.parquet')])
print(f'In-memory partitions: {df.rdd.getNumPartitions()}')
print(f'Output parquet files: {n_files}')   # one per partition

print('=== coalesce(1) write — single output file ===')
coalesce1_path = f'{BASE}/coalesce1'
df.coalesce(1).write.mode('overwrite').parquet(coalesce1_path)
n_files_c1 = len([f for f in os.listdir(coalesce1_path) if f.endswith('.parquet')])
print(f'Output parquet files: {n_files_c1}')   # 1

print('=== repartition(4).write — 4 balanced output files ===')
repart4_path = f'{BASE}/repart4'
df.repartition(4).write.mode('overwrite').parquet(repart4_path)
n_files_r4 = len([f for f in os.listdir(repart4_path) if f.endswith('.parquet')])
print(f'Output parquet files: {n_files_r4}')   # 4

print('=== write.partitionBy — directory per category value ===')
partby_path = f'{BASE}/partby_cat'
df.write.mode('overwrite').partitionBy('category').parquet(partby_path)
dirs = [d for d in os.listdir(partby_path) if os.path.isdir(f'{partby_path}/{d}')]
print(f'Directories created (one per category): {sorted(dirs)}')
# Files inside each directory
for d in sorted(dirs):
    fcount = len([f for f in os.listdir(f'{partby_path}/{d}') if f.endswith('.parquet')])
    print(f'  {d}/: {fcount} file(s)')

print('=== repartition(4, category).write — 4 files, NOT by category dir ===')
repart_cat_path = f'{BASE}/repart_cat'
df.repartition(4, 'category').write.mode('overwrite').parquet(repart_cat_path)
n_files_rc = len([f for f in os.listdir(repart_cat_path) if f.endswith('.parquet')])
dirs_rc    = [d for d in os.listdir(repart_cat_path) if os.path.isdir(f'{repart_cat_path}/{d}')]
print(f'Output parquet files: {n_files_rc}')   # 4
print(f'Sub-directories:      {dirs_rc}')       # [] — no category= dirs

In [ ]:
# Cell 5: spark.sql.shuffle.partitions and performance patterns

print('=== spark.sql.shuffle.partitions controls post-shuffle partitions ===')
print(f'Current setting: {spark.conf.get("spark.sql.shuffle.partitions")}')

# After a shuffle (groupBy, join, etc.), Spark creates this many partitions
df_grouped = df.groupBy('category').sum('amount')
print(f'Post-groupBy partition count: {df_grouped.rdd.getNumPartitions()}')
# = spark.sql.shuffle.partitions (8 in this session)

print('=== Lower the shuffle partition count for small DataFrames ===')
# 200 (default) is too many for a small dataset — lots of empty partitions
spark.conf.set('spark.sql.shuffle.partitions', '4')
df_grouped2 = df.groupBy('category').sum('amount')
print(f'Post-groupBy after setting to 4: {df_grouped2.rdd.getNumPartitions()}')  # 4
df_grouped2.show()

# Reset
spark.conf.set('spark.sql.shuffle.partitions', '8')

print('=== Pattern: repartition before a join to avoid shuffle ===')
# If two large DataFrames will be joined on the same key, pre-partition both
# so the join does not need to shuffle (saves a full shuffle stage)
left  = df.repartition(4, 'category')
right = df.groupBy('category').agg(F.sum('amount').alias('total')) \
          .repartition(4, 'category')

joined = left.join(right, on='category', how='inner')
joined.show(5)
print(f'Joined partitions: {joined.rdd.getNumPartitions()}')

print('=== Pattern: coalesce before write to reduce small-file problem ===')
# 200 post-shuffle partitions → 200 tiny output files (small-file problem)
# Collapse to a reasonable number before writing
n_target = 4
small_file_path = f'{BASE}/after_coalesce'
df_grouped.coalesce(n_target).write.mode('overwrite').parquet(small_file_path)
n_out = len([f for f in os.listdir(small_file_path) if f.endswith('.parquet')])
print(f'Output files after coalesce({n_target}): {n_out}')   # n_target

spark.stop()
print('Done.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Small-file problem — 200 empty post-shuffle partitions</summary>

**Situation:**
A daily job does a `groupBy` on a small DataFrame (10,000 rows). Spark defaults to 200 shuffle partitions, producing 200 Parquet files — most nearly empty. Downstream reads are slow because each file incurs HDFS/S3 overhead.

**Solution:**
```python
# Option 1: Lower shuffle.partitions for this job
spark.conf.set('spark.sql.shuffle.partitions', '8')
result = small_df.groupBy('region').agg(F.sum('amount'))
result.write.mode('overwrite').parquet('/output/region_summary')

# Option 2: coalesce after groupBy to reduce output file count
result = small_df.groupBy('region').agg(F.sum('amount'))
result.coalesce(4).write.mode('overwrite').parquet('/output/region_summary')

# Do NOT use repartition here — it would trigger another full shuffle
```

**Exam Sub-topic:** `spark.sql.shuffle.partitions` default is 200; `coalesce` avoids extra shuffle
</details>

<details>
<summary>Scenario 2: Skewed data — one partition much larger than others</summary>

**Situation:**
After reading a CSV, 90% of rows are in the US region. A groupBy is extremely slow because one executor is processing 90% of the data while others are idle.

**Solution:**
```python
# Read file (skewed — most rows in US region)
df = spark.read.option('header', True).option('inferSchema', True).csv('/data/sales.csv')

# Check partition sizes to confirm skew
from pyspark.sql.functions import spark_partition_id
df.withColumn('pid', spark_partition_id()) \
  .groupBy('pid').count().orderBy('count', ascending=False).show()

# repartition(20) hashes rows to 20 balanced partitions — breaks the skew
df_balanced = df.repartition(20)
result = df_balanced.groupBy('region').agg(F.sum('amount').alias('total'))
result.show()
```

**Exam Sub-topic:** `repartition(n)` does a full shuffle to balance partitions; fixes data skew
</details>

<details>
<summary>Scenario 3: Single-file output required by a legacy system</summary>

**Situation:**
A legacy reporting system expects exactly one CSV file in the output directory. Spark's default write produces multiple part files.

**Solution:**
```python
result = df.groupBy('category').agg(F.sum('amount').alias('total'))

# Collapse to single partition before writing
# coalesce is preferred over repartition(1) to avoid extra shuffle
result.coalesce(1) \
      .write \
      .mode('overwrite') \
      .option('header', True) \
      .csv('/output/category_totals')

# Result: /output/category_totals/part-00000-....csv  (only one file)
# WARNING: do not use coalesce(1) on large DataFrames — all data moves to one executor
```

**Exam Sub-topic:** `coalesce(1)` before write → single output file; avoids extra shuffle (vs `repartition(1)`)
</details>

<details>
<summary>Scenario 4: Pre-partitioning both sides of a join to avoid double shuffle</summary>

**Situation:**
Two large DataFrames are joined on `customer_id` multiple times in a pipeline. Without pre-partitioning, each join triggers a separate shuffle of both DataFrames.

**Solution:**
```python
# Repartition both DataFrames on the join key once
orders    = spark.read.parquet('/data/orders').repartition(200, 'customer_id')
customers = spark.read.parquet('/data/customers').repartition(200, 'customer_id')

# Cache after repartition so the shuffle is done only once
orders.cache()
customers.cache()

# Subsequent joins on customer_id reuse the partitioned layout
result1 = orders.join(customers, on='customer_id', how='inner')
result2 = orders.join(customers.filter(col('status') == 'active'),
                      on='customer_id', how='left')
```

**Exam Sub-topic:** `repartition(n, col)` co-locates rows by key; combined with caching avoids repeated shuffles
</details>

<details>
<summary>Scenario 5: Adaptive Query Execution automatically tunes partition count</summary>

**Situation:**
On Spark 3.x with AQE enabled, Spark automatically coalesces small post-shuffle partitions, so setting `shuffle.partitions` manually is less critical.

**Code:**
```python
# AQE is ON by default in Spark 3.x
spark.conf.get('spark.sql.adaptive.enabled')   # 'true'

# AQE coalesces small partitions automatically
spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')  # 'true'

# Even with shuffle.partitions=200, AQE may produce fewer final partitions
result = df.groupBy('region').agg(F.sum('amount'))
print(result.rdd.getNumPartitions())   # may be < 200 if AQE coalesced small partitions

# Manual repartition/coalesce still overrides AQE
result.coalesce(4).write.parquet('/output')
```

**Exam Sub-topic:** AQE coalesces small shuffle partitions automatically; `spark.sql.adaptive.enabled`
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| How to check partition count? | `df.rdd.getNumPartitions()` |
| `repartition(n)` triggers a shuffle? | **Yes** — always |
| `coalesce(n)` triggers a shuffle? | **No** — merges local partitions |
| Can `coalesce` increase partitions? | **No** — only decrease (silently caps at current count) |
| Can `repartition` increase partitions? | **Yes** |
| `repartition(n)` partition balance | **Balanced** (hash-based shuffle) |
| `coalesce(n)` partition balance | May be **skewed** (no redistribution) |
| `repartition(n, col)` effect | Rows with same `col` value go to the same partition |
| Default `shuffle.partitions` | **200** |
| How to change shuffle partitions | `spark.conf.set('spark.sql.shuffle.partitions', 'N')` |
| Single output file pattern | `df.coalesce(1).write...` |
| Why `coalesce(1)` over `repartition(1)`? | Avoids extra shuffle — cheaper |
| When to use `repartition` over `coalesce`? | When you need balanced partitions or to **increase** count |
| `write.partitionBy` vs `repartition` | `partitionBy` = storage dirs; `repartition` = memory partitions |
| Small-file problem cause | Too many shuffle partitions (default 200) for small data |
| Small-file solution | Lower `shuffle.partitions` or `coalesce` before write |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between repartition(n) and coalesce(n)?</summary>

**A:**
- **`repartition(n)`** performs a **full shuffle** to redistribute data evenly into exactly `n` balanced partitions. It can both increase and decrease the partition count. Expensive due to the shuffle.
- **`coalesce(n)`** merges existing partitions without a full shuffle. It can **only decrease** the partition count. Cheaper but may produce uneven (skewed) partition sizes.

Rule of thumb: use `coalesce` to reduce (e.g., before writing), use `repartition` to balance or increase.
</details>

<details>
<summary>Q2: You call df.coalesce(500) on a DataFrame with 4 partitions. What is the result?</summary>

**A:** The DataFrame still has **4 partitions**. `coalesce` cannot increase the partition count — it silently caps at the current number. No error is raised. To increase to 500 partitions you must use `df.repartition(500)`.
</details>

<details>
<summary>Q3: A groupBy produces 200 post-shuffle partitions for a 1,000-row DataFrame. How do you fix the small-file problem before writing?</summary>

**A:** Two options:
1. **Lower `shuffle.partitions` before the job:**
   ```python
   spark.conf.set('spark.sql.shuffle.partitions', '8')
   result = df.groupBy('region').agg(F.sum('amount'))
   result.write.parquet('/output')
   ```
2. **Use `coalesce` after the groupBy:**
   ```python
   result = df.groupBy('region').agg(F.sum('amount'))
   result.coalesce(4).write.parquet('/output')   # 4 output files instead of 200
   ```

Prefer `coalesce` over `repartition` here because it avoids an extra shuffle.
</details>

<details>
<summary>Q4: What does repartition(4, 'region') do differently from repartition(4)?</summary>

**A:**
- **`repartition(4)`** distributes rows randomly (hash of the entire row or a random hash) into 4 balanced partitions. Rows with the same `region` may end up in different partitions.
- **`repartition(4, 'region')`** hashes specifically on the `region` column. All rows with the same `region` value are guaranteed to land in the **same partition**. This is useful before a join or groupBy on `region` — it eliminates the join-time shuffle.
</details>

<details>
<summary>Q5: Why is coalesce(1) preferred over repartition(1) when writing a single output file?</summary>

**A:** Both produce a single output file. However:
- **`repartition(1)`** always performs a **full shuffle** — all data is moved across the network to one executor.
- **`coalesce(1)`** tries to merge partitions that are already on the same executor, avoiding network transfer where possible. It is cheaper.

For production: only use `coalesce(1)` on small DataFrames — writing all data through a single partition is a bottleneck for large datasets.
</details>